# SPRO Advantage Computation Test

Quick test notebook to verify advantage computation is stable.

**Key metrics to check:**
- Advantages should be in range [-5, 5] (clipped)
- Mean should be ~0 after normalization
- Std should be ~1 after normalization
- Policy loss should be in range [-5, 5] (not -700 to +200!)

## 0. Setup - Clone Repo

In [ ]:
import os

# Clone repo if not already present
if not os.path.exists('JB_mech'):
    !git clone https://github.com/ChuloIva/JB_mech.git
    print("Repo cloned!")
else:
    # Pull latest changes
    !cd JB_mech && git pull
    print("Repo already exists, pulled latest.")

# Add src to path
import sys
sys.path.insert(0, 'JB_mech/src')

print("Path configured!")

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

from spro import SPROConfig
from spro.spro_core import compute_spro_advantages, compute_spro_advantages_v2

print("Imports successful!")

## 1. Test with Synthetic Data

In [ ]:
# Simulate a batch of 8 episodes with varying sequence lengths
batch_size = 8
max_seq_len = 2000  # Typical generated sequence length
response_start = 500  # Prompt takes ~500 tokens

# Simulate outcome rewards (score 2 = -0.2, score 3 = 0.5, score 4 = 1.0)
# Typical distribution: mostly 2s, occasional 3 or 4
outcome_rewards = torch.tensor([-0.2, -0.2, -0.2, -0.2, -0.2, -0.2, 0.5, -0.2])
print(f"Outcome rewards: {outcome_rewards.tolist()}")
print(f"  Mean: {outcome_rewards.mean():.4f}, Std: {outcome_rewards.std():.4f}")

In [ ]:
# Simulate log-prob ratios (typically small values, can be positive or negative)
# Early in training, policy ≈ reference, so log_ratios ≈ 0
# As training progresses, log_ratios diverge

torch.manual_seed(42)
log_ratios = torch.randn(batch_size, max_seq_len) * 0.1  # Small variance

# Create response mask
response_mask = torch.zeros(batch_size, max_seq_len)
response_mask[:, response_start:] = 1.0

# Zero out prompt tokens in log_ratios
log_ratios[:, :response_start] = 0.0

print(f"Log ratios shape: {log_ratios.shape}")
valid_lr = log_ratios[response_mask.bool()]
print(f"Valid log ratios: mean={valid_lr.mean():.4f}, std={valid_lr.std():.4f}, min={valid_lr.min():.4f}, max={valid_lr.max():.4f}")

In [ ]:
# Test OLD computation (cumsum - this was the problem)
def compute_spro_advantages_old(outcome_rewards, log_ratios, response_mask):
    """OLD buggy version for comparison."""
    batch_size, seq_len = log_ratios.shape
    
    # Outcome advantage
    outcome_mean = outcome_rewards.mean()
    outcome_std = outcome_rewards.std() + 1e-8
    outcome_advantage = (outcome_rewards - outcome_mean) / outcome_std
    outcome_expanded = outcome_advantage.unsqueeze(1).expand(-1, seq_len)
    
    # OLD: cumulative sum (grows unbounded!)
    cumulative = torch.cumsum(log_ratios, dim=1)
    
    # MSA without normalization
    msa = torch.zeros_like(cumulative)
    for t in range(seq_len):
        valid_mask = response_mask[:, t].bool()
        if valid_mask.sum() > 0:
            mean_t = cumulative[valid_mask, t].mean()
            msa[:, t] = cumulative[:, t] - mean_t
    
    advantages = outcome_expanded + msa
    advantages = advantages * response_mask
    
    return advantages

# Compute OLD advantages
old_adv = compute_spro_advantages_old(outcome_rewards, log_ratios, response_mask)
valid_old = old_adv[response_mask.bool()]
print("OLD Advantages (buggy):")
print(f"  Mean: {valid_old.mean():.4f}")
print(f"  Std: {valid_old.std():.4f}")
print(f"  Min: {valid_old.min():.4f}")
print(f"  Max: {valid_old.max():.4f}")
print(f"  Range: {valid_old.max() - valid_old.min():.4f}")

In [ ]:
# Test NEW computation (fixed)
config = SPROConfig()

new_adv = compute_spro_advantages(
    outcome_rewards,
    log_ratios,
    response_mask,
    msa_scale=config.msa_scale,
    advantage_clip=config.advantage_clip,
    normalize_final=config.normalize_advantages,
)

valid_new = new_adv[response_mask.bool()]
print("NEW Advantages (fixed):")
print(f"  Mean: {valid_new.mean():.4f}")
print(f"  Std: {valid_new.std():.4f}")
print(f"  Min: {valid_new.min():.4f}")
print(f"  Max: {valid_new.max():.4f}")
print(f"  Range: {valid_new.max() - valid_new.min():.4f}")

In [ ]:
# Test V2 (GAE-style) computation
v2_adv = compute_spro_advantages_v2(
    outcome_rewards,
    log_ratios,
    response_mask,
    gamma=0.99,
    advantage_clip=config.advantage_clip,
)

valid_v2 = v2_adv[response_mask.bool()]
print("V2 Advantages (GAE-style):")
print(f"  Mean: {valid_v2.mean():.4f}")
print(f"  Std: {valid_v2.std():.4f}")
print(f"  Min: {valid_v2.min():.4f}")
print(f"  Max: {valid_v2.max():.4f}")
print(f"  Range: {valid_v2.max() - valid_v2.min():.4f}")

## 2. Test with Extreme Log Ratios (Simulating Later Training)

In [ ]:
# Later in training, log ratios can have higher variance
torch.manual_seed(42)
extreme_log_ratios = torch.randn(batch_size, max_seq_len) * 0.5  # Higher variance
extreme_log_ratios[:, :response_start] = 0.0

valid_extreme = extreme_log_ratios[response_mask.bool()]
print(f"Extreme log ratios: mean={valid_extreme.mean():.4f}, std={valid_extreme.std():.4f}")
print(f"  Min={valid_extreme.min():.4f}, Max={valid_extreme.max():.4f}")

In [ ]:
# OLD computation with extreme values
old_extreme = compute_spro_advantages_old(outcome_rewards, extreme_log_ratios, response_mask)
valid_old_ext = old_extreme[response_mask.bool()]
print("OLD with extreme log ratios:")
print(f"  Mean: {valid_old_ext.mean():.4f}")
print(f"  Std: {valid_old_ext.std():.4f}")
print(f"  Min: {valid_old_ext.min():.4f}")
print(f"  Max: {valid_old_ext.max():.4f}")
print(f"  Range: {valid_old_ext.max() - valid_old_ext.min():.4f}")
print(f"  >>> THIS EXPLAINS THE -700 to +200 POLICY LOSS! <<<")

In [ ]:
# NEW computation with extreme values
new_extreme = compute_spro_advantages(
    outcome_rewards, extreme_log_ratios, response_mask,
    msa_scale=0.1, advantage_clip=5.0, normalize_final=True
)
valid_new_ext = new_extreme[response_mask.bool()]
print("NEW with extreme log ratios:")
print(f"  Mean: {valid_new_ext.mean():.4f}")
print(f"  Std: {valid_new_ext.std():.4f}")
print(f"  Min: {valid_new_ext.min():.4f}")
print(f"  Max: {valid_new_ext.max():.4f}")
print(f"  Range: {valid_new_ext.max() - valid_new_ext.min():.4f}")
print(f"  >>> STABLE! <<<")

## 3. Simulate PPO Loss Computation

In [ ]:
def simulate_ppo_loss(advantages, response_mask, clip_range=0.2):
    """
    Simulate PPO loss computation.
    
    In real PPO:
    - ratio = exp(new_log_prob - old_log_prob) ≈ 1 early in training
    - loss = -min(ratio * adv, clip(ratio) * adv)
    
    Since ratio ≈ 1 early in training, loss ≈ -advantages
    """
    # Simulate ratio close to 1
    ratio = torch.ones_like(advantages) + torch.randn_like(advantages) * 0.05
    
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1 - clip_range, 1 + clip_range) * advantages
    
    # PPO loss (per token)
    token_loss = -torch.min(surr1, surr2) * response_mask
    
    # Aggregate
    total_loss = token_loss.sum() / (response_mask.sum() + 1e-8)
    
    return total_loss.item()

print("PPO Loss Simulation:")
print(f"  OLD advantages: {simulate_ppo_loss(old_extreme, response_mask):.4f}")
print(f"  NEW advantages: {simulate_ppo_loss(new_extreme, response_mask):.4f}")
print(f"  V2 advantages: {simulate_ppo_loss(compute_spro_advantages_v2(outcome_rewards, extreme_log_ratios, response_mask), response_mask):.4f}")

## 4. Hyperparameter Sensitivity

In [ ]:
# Test different msa_scale values
print("MSA Scale Sensitivity:")
for scale in [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]:
    adv = compute_spro_advantages(
        outcome_rewards, extreme_log_ratios, response_mask,
        msa_scale=scale, advantage_clip=5.0, normalize_final=True
    )
    valid = adv[response_mask.bool()]
    loss = simulate_ppo_loss(adv, response_mask)
    print(f"  scale={scale:.2f}: std={valid.std():.4f}, range=[{valid.min():.2f}, {valid.max():.2f}], loss={loss:.4f}")

In [ ]:
# Test different advantage_clip values
print("\nAdvantage Clip Sensitivity:")
for clip in [1.0, 2.0, 5.0, 10.0, 20.0]:
    adv = compute_spro_advantages(
        outcome_rewards, extreme_log_ratios, response_mask,
        msa_scale=0.1, advantage_clip=clip, normalize_final=True
    )
    valid = adv[response_mask.bool()]
    loss = simulate_ppo_loss(adv, response_mask)
    print(f"  clip={clip:.1f}: std={valid.std():.4f}, range=[{valid.min():.2f}, {valid.max():.2f}], loss={loss:.4f}")

## 5. Summary

**Changes made:**
1. Removed cumulative sum (was causing unbounded growth)
2. Added per-timestep normalization (mean AND std)
3. Added MSA scaling (default 0.1)
4. Added final advantage normalization
5. Added advantage clipping (default 5.0)

**Expected results:**
- Advantages in range [-5, 5]
- Mean ≈ 0
- Std ≈ 1
- Policy loss in range [-5, 5]

In [ ]:
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"\nOLD (buggy):")
print(f"  Advantage range: [{valid_old_ext.min():.2f}, {valid_old_ext.max():.2f}]")
print(f"  Policy loss: {simulate_ppo_loss(old_extreme, response_mask):.2f}")
print(f"\nNEW (fixed):")
print(f"  Advantage range: [{valid_new_ext.min():.2f}, {valid_new_ext.max():.2f}]")
print(f"  Policy loss: {simulate_ppo_loss(new_extreme, response_mask):.2f}")
print(f"\n>>> Policy loss reduced from ~{abs(simulate_ppo_loss(old_extreme, response_mask)):.0f} to ~{abs(simulate_ppo_loss(new_extreme, response_mask)):.1f} <<<")